# 08 - RAG Using LangChain


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Build RAG using LangChain components.
- Understand LCEL-style composition.
- Add retrieval and answer evaluation.
- Introduce Ragas metrics conceptually and optionally in code.

## Core AI concept

LangChain separates application pieces into composable components: prompts, models, retrievers, parsers, and tools. The key learning goal is not memorizing chain syntax; it is knowing what each component does and how to test it independently.

## Architecture / mental model

```text
Question → Retriever → Context
             ↓
        ChatPromptTemplate
             ↓
            LLM
             ↓
       Output parser
             ↓
           Answer

```

## Connection to the supplied learning material

The supplied RAG notebook already combines retrieval, prompt construction, model generation, source-aware context, reranking, and basic evaluation. This version makes those pieces more modular.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU langchain langchain-openai langchain-chroma langchain-text-splitters ragas

### Build a small LangChain RAG chain

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = [
    Document(page_content="RAG retrieves external context before generation.", metadata={"source": "rag"}),
    Document(page_content="Embeddings convert text into vectors used for similarity search.", metadata={"source": "embeddings"}),
]

chunks = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30
).split_documents(docs)

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

store = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(model=OPENAI_EMBED_MODEL),
    collection_name="langchain_rag",
    persist_directory="./lc_rag_chroma",
)
retriever = store.as_retriever(search_kwargs={"k": 3})

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Answer only from context.\n\nContext:\n{context}\n\nQuestion:\n{question}"
)
llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)

def run_rag(question):
    retrieved = retriever.invoke(question)
    context = "\n\n".join(
        f"[{d.metadata.get('source')}] {d.page_content}" for d in retrieved
    )
    answer = (prompt | llm | StrOutputParser()).invoke({
        "context": context, "question": question
    })
    return answer, retrieved

# print(run_rag("What are embeddings?")[0])

### Lightweight retrieval evaluation

In [ ]:
test_set = [
    {"question": "What is RAG?", "expected_source": "rag"},
    {"question": "What are embeddings?", "expected_source": "embeddings"},
]

def hit_at_k(retrieved, expected_source):
    return int(any(d.metadata.get("source") == expected_source for d in retrieved))

scores = []
for item in test_set:
    answer, retrieved = run_rag(item["question"])
    score = hit_at_k(retrieved, item["expected_source"])
    scores.append(score)
    print(item["question"], "hit:", score, "answer:", answer[:120])

print("Hit rate:", sum(scores) / len(scores))

### Optional Ragas direction

In [ ]:
# Current Ragas exposes RAG metrics such as:
# Context Precision, Context Recall, Response Relevancy,
# Faithfulness, and multimodal variants.
#
# For a production evaluation project, build a dataset containing:
# user_input, response, retrieved_contexts, and (when available) reference.

## Evaluation

Use retrieval hit rate/Recall@K for the retriever. For generated answers, measure faithfulness, response relevancy, context precision, and context recall. Ragas currently documents these as core RAG metrics.

## Improvement ideas

Separate retrieval evaluation from answer evaluation. A bad answer caused by bad retrieval should not be fixed by blindly changing the LLM prompt.

## Practice exercises

1. Add five more gold questions.
2. Add an answer-reference column.
3. Compare two retrievers and two chunk sizes.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create

- Ragas available metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/